# Typed tensor experimental

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import inspect
from builtins import bool as _bool
from builtins import int as _int
from dataclasses import dataclass
from typing import (
    Any,
    Dict,
    Generic,
    List,
    Optional,
    Tuple,
    Union,
    get_args,
    get_origin,
)
from typing import Literal as L

import torch
from torch._C import _TensorMeta
from typing_extensions import Self, Type, TypeAlias, TypeVar

from torchwrench.core import dtype_enum_v2 as dtypes
from torchwrench.core import device_enum as devices
from torchwrench.core.device_enum import (
    CPUDeviceType,
    DeviceBase,
    device_cls_to_torch_device,
)
from torchwrench.core.dtype_enum_v2 import (
    DTypeBase,
    dtype_cls_to_dtype,
    dtype_to_dtype_cls,
)
from torchwrench.core.make import DeviceLike, DTypeLike, as_device, as_dtype

In [2]:
_0DShape = Tuple[()]
_1DShape = Tuple[_int]
_2DShape = Tuple[_int, _int]
_3DShape = Tuple[_int, _int, _int]
_4DShape = Tuple[_int, _int, _int, _int]


_AnyShape: TypeAlias = Any
_AnyDType: TypeAlias = Any
_AnyDevice: TypeAlias = Any

_ShapeGenericType: TypeAlias = Union[Tuple[_int, ...], _AnyShape]
_DTypeGenericType: TypeAlias = Union[DTypeBase, _AnyDType]
_DeviceGenericType: TypeAlias = Union[DeviceBase, _AnyDevice]


T_Shape = TypeVar(
    "T_Shape",
    bound=_ShapeGenericType,
    default=_AnyShape,
    covariant=True,
)
T_DType = TypeVar(
    "T_DType",
    bound=_DTypeGenericType,
    default=_AnyDType,
    covariant=True,
)
T_Device = TypeVar(
    "T_Device",
    bound=_DeviceGenericType,
    default=CPUDeviceType,
    covariant=True,
)

T_Shape2 = TypeVar(
    "T_Shape2",
    bound=_ShapeGenericType,
    default=_AnyShape,
    covariant=True,
)
T_DType2 = TypeVar(
    "T_DType2",
    bound=_DTypeGenericType,
    default=_AnyDType,
    covariant=True,
)
T_Device2 = TypeVar(
    "T_Device2",
    bound=_DeviceGenericType,
    default=_AnyDevice,
    covariant=True,
)

T_Shape3 = TypeVar(
    "T_Shape3",
    bound=_ShapeGenericType,
    default=_AnyShape,
    covariant=True,
)
T_DType3 = TypeVar(
    "T_DType3",
    bound=_DTypeGenericType,
    default=_AnyDType,
    covariant=True,
)
T_Device3 = TypeVar(
    "T_Device3",
    bound=_DeviceGenericType,
    default=_AnyDevice,
    covariant=True,
)

T_Axis0 = TypeVar(
    "T_Axis0",
    bound=_int,
    default=_int,
    covariant=True,
)
T_Axis1 = TypeVar(
    "T_Axis1",
    bound=_int,
    default=_int,
    covariant=True,
)
T_Axis2 = TypeVar(
    "T_Axis2",
    bound=_int,
    default=_int,
    covariant=True,
)

In [3]:
    # @classmethod
    # def __class_getitem__(cls, gen_args):
    #     # # Create a specialized subclass: Box[int], Box[str], etc.
    #     # name = f"{cls.__name__}[" + ",".join(map(str, args)) + "]"
    #     # print(f"{cls=}")
    #     # print(f"{args=}")
    #     # return type(name, (cls,), {"__orig_bases__": [TTensor], "__custom__": args})    # Create a specialized subclass with proper generic parameters

    #     # # Create a specialized subclass with proper generic parameters
    #     name = f"{cls.__name__}[" + ",".join(map(str, gen_args)) + "]"

    #     # if cls is TTensor:
    #     #     generic_args = args[0]
    #     # else:
    #     #     generic_args = ()
    #     # breakpoint()

    #     # TODO: rm
    #     # print(f"{generic_args=}")
    #     # breakpoint()

    #     # # Get the current class's __orig_bases__ to extract the shape constraint
    #     # if hasattr(cls, "__orig_bases__"):
    #     #     orig_bases = cls.__orig_bases__  # type: ignore
    #     #     # Find TTensor base
    #     #     ttensor_base = None
    #     #     for base in orig_bases:
    #     #         if get_origin(base) is TTensor:
    #     #             ttensor_base = base
    #     #             break
    #     #     base_args = get_args(ttensor_base) if ttensor_base else ()
    #     # else:
    #     #     base_args = ()

    #     # Extract shape from parent, merge with new args
    #     # shape_arg = base_args[0] if len(base_args) > 0 else _AnyShape
    #     # dtype_arg = args[0] if len(args) > 0 else _AnyDType
    #     # device_arg = args[1] if len(args) > 1 else _AnyDevice

    #     # Create a wrapper class that stores generics and delegates __new__ to parent
    #     # class SpecializedTensor:  # type: ignore
    #     #     __new__ = cls.__new__
    #     #     # __generic_params__ = (shape_arg, dtype_arg, device_arg)
    #     #     __generic_params__ = args[0]

    #     # SpecializedTensor.__name__ = name
    #     # SpecializedTensor.__qualname__ = name

    #     # print(f"{SpecializedTensor.__generic_params__=}")
    #     # breakpoint()
    #     # return SpecializedTensor  # type: ignore

    #     # full_generics = _resolve_ttensors_generics(cls, args)
    #     print(f"Class getitem:")
    #     print(f"{cls=}")
    #     print(f"{gen_args=}")
    #     # print(f"{full_generics=}")

    #     # class SpecializedTensor(cls):
    #     #     __new__ = cls.__new__
    #     #     __generic_params__ = generic_args

    #     # SpecializedTensor.__name__ = name
    #     # SpecializedTensor.__qualname__ = name

    #     base_cls = cls
    #     cls = type(name, (cls,), {"__orig__": TTensor})

    #     if base_cls is TTensor:
    #         if len(gen_args) > 0:
    #             gen_shape = gen_args[0]
    #         else:
    #             gen_shape = T_Shape

    #         if len(gen_args) > 1:
    #             gen_dtype = gen_args[1]
    #         else:
    #             gen_dtype = T_DType

    #         if len(gen_args) > 2:
    #             gen_device = gen_args[2]
    #         else:
    #             gen_device = T_Device

    #         if not hasattr(cls, "__gen_shape__") or isinstance(
    #             cls.__gen_shape__, TypeVar
    #         ):
    #             cls.__gen_shape__ = gen_shape  # type: ignore

    #         if not hasattr(cls, "__gen_dtype__") or isinstance(
    #             cls.__gen_dtype__, TypeVar
    #         ):
    #             cls.__gen_dtype__ = gen_dtype  # type: ignore

    #         if not hasattr(cls, "__gen_device__") or isinstance(
    #             cls.__gen_device__, TypeVar
    #         ):
    #             cls.__gen_device__ = gen_device  # type: ignore

    #         return super().__class_getitem__((gen_args,))  # type: ignore

    #     else:
    #         for arg in gen_args:
    #             if (
    #                 not hasattr(cls, "__gen_shape__")
    #                 or isinstance(cls.__gen_shape__, TypeVar)
    #             ) and get_origin(arg) is tuple:
    #                 cls.__gen_shape__ = arg
    #             if (
    #                 (
    #                     not hasattr(cls, "__gen_dtype__")
    #                     or isinstance(cls.__gen_dtype__, TypeVar)
    #                 )
    #                 and isinstance(arg, type)
    #                 and issubclass(arg, DTypeBase)
    #             ):
    #                 cls.__gen_dtype__ = arg
    #             if (
    #                 (
    #                     not hasattr(cls, "__gen_device__")
    #                     or isinstance(cls.__gen_device__, TypeVar)
    #                 )
    #                 and isinstance(arg, type)
    #                 and issubclass(arg, DeviceBase)
    #             ):
    #                 cls.__gen_device__ = arg

    #     print(f"{cls.__gen_shape__=}")
    #     print(f"{cls.__gen_dtype__=}")
    #     print(f"{cls.__gen_device__=}")

    #     # breakpoint()
    #     return super().__class_getitem__((gen_args,))  # type: ignore

In [4]:
# class _ndim_descriptor:
#     @overload
#     def __get__(
#         self,
#         instance: "TTensor[_0DShape, Any, Any]",
#         owner: Any,
#     ) -> L[0]: ...
#     @overload
#     def __get__(
#         self,
#         instance: "TTensor[_1DShape, Any, Any]",
#         owner: Any,
#     ) -> L[1]: ...
#     @overload
#     def __get__(
#         self,
#         instance: "TTensor[_2DShape, Any, Any]",
#         owner: Any,
#     ) -> L[2]: ...
#     @overload
#     def __get__(
#         self,
#         instance: "TTensor[_3DShape, Any, Any]",
#         owner: Any,
#     ) -> L[3]: ...
#     @overload
#     def __get__(
#         self,
#         instance: "TTensor[_4DShape, Any, Any]",
#         owner: Any,
#     ) -> L[4]: ...
#     @overload
#     def __get__(
#         self,
#         instance: "TTensor[Any, Any, Any]",
#         owner: Any,
#     ) -> _int: ...

#     def __get__(self, instance: object, owner: Any) -> _int:
#         raise NotImplementedError


# def _resolve_ttensors_generics(cls, args):
#     # Find TTensor[...] base
#     for base in getattr(cls, "__orig_bases__", ()):
#         if get_origin(base) is TTensor:
#             base_args = list(get_args(base))  # (_2DShape, T_DType, T_Device)
#             break
#     else:
#         base_args = [None, None, None]

#     # Replace TypeVars with provided args
#     resolved = []
#     arg_iter = iter(args)

#     for a in base_args:
#         if isinstance(a, TypeVar):
#             resolved.append(next(arg_iter))
#         else:
#             resolved.append(a)

#     return tuple(resolved)


In [5]:
@dataclass
class _TTensorGenerics:
    shape: Union[Tuple[Optional[_int], ...], None] = None
    dtype: Union[torch.dtype, None] = None
    device: Union[torch.device, None] = None

    def is_compatible_with_tensor(self, x: torch.Tensor) -> _bool:
        return (
            self.is_compatible_with_shape(x.shape)
            and self.is_compatible_with_dtype(x.dtype)
            and self.is_compatible_with_device(x.device)
        )

    def is_compatible_with_tensor_generics(self, x: Self) -> _bool:
        return (
            (x.shape is None or self.is_compatible_with_shape(x.shape))
            and (x.dtype is None or self.is_compatible_with_dtype(x.dtype))
            and (x.device is None or self.is_compatible_with_device(x.device))
        )

    def is_compatible_with_shape(self, shape: Tuple[Optional[_int], ...]) -> _bool:
        if self.shape is None:
            return True

        if len(get_args(self.shape)) != len(shape):
            return False

        valid = [
            (s1 is None) or (s2 is None) or (s1 == s2)
            for s1, s2 in zip(get_args(self.shape), shape)
        ]
        return all(valid)

    def is_compatible_with_dtype(self, dtype: Union[torch.dtype, DTypeBase]) -> _bool:
        """Check if the dtype is compatible with the generic dtype."""
        if self.dtype is None or dtype is None:
            return True

        if isinstance(dtype, torch.dtype):
            dtype_enum = dtype_to_dtype_cls(dtype)
        else:
            dtype_enum = dtype

        return self.dtype == dtype_enum

    def is_compatible_with_device(
        self,
        device: Union[torch.device, str, _int],
    ) -> _bool:
        if self.device is None:
            return True

        return as_device(self.device) == as_device(device)

    @property
    def ndim(self) -> Optional[_int]:
        """Get the number of dimensions from the shape generic parameter."""
        if self.shape is None:
            return None

        generic_shape_args = get_args(self.shape)
        if len(generic_shape_args) == 2 and generic_shape_args[1] is ...:
            return None

        if isinstance(self.shape, tuple):
            return len(self.shape)

        msg = f"Internal error: cannot get ndim for {self.shape=}."
        raise NotImplementedError(msg)


def _generic_shape_to_shape(
    gen_shape: _ShapeGenericType,
) -> Optional[Tuple[Optional[_int], ...]]:
    if isinstance(gen_shape, TypeVar) or gen_shape is _AnyShape:
        return None

    shape_types = get_args(gen_shape)
    # print(f"DEBUG: {shape_types=}; {gen_shape=}")
    if shape_types == ((),):
        return ()
    else:
        return tuple(
            get_args(s)[0] if (get_origin(s) is L) else None
            for s in shape_types
        )


def _generic_dtype_to_dtype(gen_dtype: _DTypeGenericType) -> Optional[torch.dtype]:
    if isinstance(gen_dtype, TypeVar) or gen_dtype is _AnyDType:
        return None

    if isinstance(gen_dtype, type) and DTypeBase in gen_dtype.__mro__:
        return dtype_cls_to_dtype(gen_dtype)

    msg = f"Invalid argument {gen_dtype=} (expected DTypeEnum or None)."
    raise TypeError(msg)


def _generic_device_to_device(
    gen_device: _DeviceGenericType,
) -> Optional[torch.device]:
    if isinstance(gen_device, TypeVar) or gen_device is _AnyDevice:
        return None
    else:
        return device_cls_to_torch_device(gen_device)


def _cls_to_generics(cls: type) -> _TTensorGenerics:
    """Get the generic parameters of a TTensor subclass."""

    if (
        hasattr(cls, "__gen_shape__")
        and hasattr(cls, "__gen_dtype__")
        and hasattr(cls, "__gen_device__")
    ):
        args = (cls.__gen_shape__, cls.__gen_dtype__, cls.__gen_device__)
    elif cls is TTensor or get_origin(cls) is TTensor:
        args = get_args(cls)
    elif hasattr(cls, "__orig_bases__"):
        orig_bases = cls.__orig_bases__  # type: ignore
        args = None
        for base in orig_bases:
            if get_origin(base) is TTensor:
                args = get_args(base)
                break
        if args is None:
            msg = f"Invalid argument {cls=}. (expected TTensor or subclass or TTensor)"
            raise TypeError(msg)
    else:
        msg = f"Invalid argument {cls=}. (expected TTensor or subclass or TTensor)"
        raise TypeError(msg)

    if len(args) < 3:
        missing = (_AnyShape, _AnyDType, _AnyDevice)[len(args) :]
        args = args + missing

    if len(args) != 3:
        msg = (
            f"Expected 3 generic parameters for TTensor, but got {len(args)} in {cls}."
        )
        raise TypeError(msg)

    generic_shape, generic_dtype, generic_device = args
    # TODO: rm
    # print(f"{cls=}")
    # print(f"{generic_shape=}")
    # print(f"{generic_dtype=}")
    # print(f"{generic_device=}")
    # breakpoint()

    shape = _generic_shape_to_shape(generic_shape)
    dtype = _generic_dtype_to_dtype(generic_dtype)
    device = _generic_device_to_device(generic_device)

    return _TTensorGenerics(
        shape=shape,
        dtype=dtype,
        device=device,
    )


def _get_generics(cls_or_instance: Any) -> _TTensorGenerics:
    """Get the generic parameters of a TTensor subclass or instance."""
    if not isinstance(cls_or_instance, type):
        cls_or_instance = type(cls_or_instance)
    return _cls_to_generics(cls_or_instance)


class _TTensorMeta(_TensorMeta):
    def __instancecheck__(self, instance: Any) -> _bool:
        """Called method to check isinstance(instance, TTensor)"""
        if not isinstance(instance, torch.Tensor):
            return False

        gen = _cls_to_generics(self)
        return gen.is_compatible_with_tensor(instance)

    def __subclasscheck__(self, subclass: Any) -> _bool:
        """Called method to check issubclass(subclass, TTensor)"""
        self_generics = _cls_to_generics(self)
        other_generics = _cls_to_generics(subclass)
        return self_generics.is_compatible_with_tensor_generics(other_generics)

In [16]:
class TTensor(
    Generic[T_Shape, T_DType, T_Device],
    torch.Tensor,
    metaclass=_TTensorMeta,
):
    _DEFAULT_DTYPE: Optional[Type[DTypeBase]] = None

    @classmethod
    def __class_getitem__(cls, gen_args):
        # name = f"{cls.__name__}[" + ", ".join(map(repr, gen_args)) + "]"
        # name = f"{cls.__name__}[{gen_args}]"
        # print(f"__class_getitem__: {name}")

        # class SpecializedTensor():  # type: ignore
        #     __new__ = cls.__new__
        #     __generic_params__ = gen_args
        #     # __class_getitem__ = cls.__class_getitem__

        # SpecializedTensor.__name__ = name
        # SpecializedTensor.__qualname__ = name
        # return SpecializedTensor

        result = super().__class_getitem__(gen_args)  # type: ignore
        # print(f"__class_getitem__: {cls}; {gen_args}")
#         resolved_args = get_args(result)
#         generics = [basecls for basecls in cls.__orig_bases__ if get_origin(basecls) is Generic]  # type: ignore
#         # print(f"{generics=}")
#         if len(generics) == 1:
#             generic_names = [typevar.__name__ for typevar in get_args(generics[0]) if typevar in (T_Shape, T_DType, T_Device)]
#         else:
#             generic_names = []
# # 
#         # attrs = [getattr(a, '__name__', f"{i}") for i, (a, gen_name) in enumerate(zip(resolved_args, generic_names))]
#         gen_args_dict = dict(zip(generic_names, resolved_args))
#         print(f"__class_getitem__: {cls}; {gen_args}; {gen_args_dict=}")

#         if getattr(result, "_shape", None) is None:
#             setattr(result, "_shape", _generic_shape_to_shape(gen_args_dict.get("T_Shape", Any)))
#         if getattr(result, "_dtype", None) is None:
#             setattr(result, "_dtype", _generic_dtype_to_dtype(gen_args_dict.get("T_DType", Any)))
#         if getattr(result, "_device", None) is None:
#             setattr(result, "_device", _generic_device_to_device(gen_args_dict.get("T_Device", Any)))
#         print("__class_getitem__:", result._shape, result._dtype, result._device, f"{id(result)}")

        return result

    def __new__(
        cls: "Type[TTensor[T_Shape, T_DType, T_Device]]",
        *args: Any,
        dtype: DTypeLike = None,
        device: DeviceLike = None,
        memory_format: Union[torch.memory_format, None] = None,
        out: Union[torch.Tensor, None] = None,
        layout: Union[torch.layout, None] = None,
        pin_memory: Union[_bool, None] = False,
        requires_grad: Union[_bool, None] = False,
    ) -> "TTensor[T_Shape, T_DType, T_Device]":
        dtype = as_dtype(dtype)
        device = as_device(device)

        print("__new__:", cls)

        # gen = _get_generics(cls)  # type: ignore
        gen = _TTensorGenerics(None, None, None)  # TODO: rm
        cls_dtype = gen.dtype
        cls_ndim = gen.ndim
        cls_shape = gen.shape
        cls_device = gen.device

        # print(f"{gen=}")
        # print(f"{getattr(cls, '__gen_shape__', None)=}")
        # print(f"{getattr(cls, '__gen_dtype__', None)=}")
        # print(f"{getattr(cls, '__gen_device__', None)=}")
        # # breakpoint()

        # Sanity checks for dtype
        if dtype is None:
            if cls_dtype is not None:
                dtype = cls_dtype
            elif getattr(cls, "_DEFAULT_DTYPE", None) is not None:
                dtype = dtype_cls_to_dtype(cls._DEFAULT_DTYPE)

        elif cls_dtype is None:
            if not gen.is_compatible_with_dtype(dtype):
                msg = f"Invalid argument {dtype=} for {cls.__name__}."
                raise ValueError(msg)

        elif dtype == cls_dtype:
            pass
        else:
            msg = (
                f"Invalid argument {dtype=} for {cls.__name__}. (expected {cls_dtype})"
            )
            raise ValueError(msg)

        # Sanity checks for device
        if device is None:
            if cls_device is not None:
                device = cls_device
        elif cls_device is None or device == cls_device:
            pass
        else:
            msg = f"Invalid argument {device=} for {cls.__name__}. (expected {cls_device})"
            raise ValueError(msg)

        # Sanity checks for data and ndim
        is_int_args = all(isinstance(arg, _int) for arg in args)
        if cls_ndim is None:
            if len(args) == 0:
                size = (0,)
                data = None
            elif is_int_args:
                size = args
                data = None
            elif len(args) == 1 and not isinstance(args[0], _int):
                size = None
                data = args[0]
            else:
                msg = f"Invalid arguments {args=}. (expected only ints or one sequence of data)"
                raise ValueError(msg)

        elif is_int_args and cls_ndim == len(args):
            if not gen.is_compatible_with_shape(args):
                msg = f"Invalid argument {args=} for {cls.__name__}. (expected shape {cls_shape})"
                raise ValueError(msg)

            size = args
            data = None

        elif len(args) == 1:
            size = None
            data = args[0]
        elif len(args) == 0:
            size = [0] * cls_ndim
            data = None
        else:
            msg = f"Invalid arguments {args=}. (expected {cls_ndim} ints or one sequence of data)"
            raise ValueError(msg)
        del args

        # Supports older PyTorch versions
        if layout is None:
            layout = torch.strided
        if pin_memory is None:
            pin_memory = False
        if requires_grad is None:
            requires_grad = False

        if data is not None:
            result = torch.as_tensor(
                data=data,
                dtype=dtype,  # type: ignore
                device=device,
            )
            if cls_ndim is not None and result.ndim != cls_ndim:
                msg = f"Invalid number of dimension(s) for argument data in {cls.__name__}. (found {result.ndim} but expected {cls_ndim})"
                raise ValueError(msg)
            return result  # type: ignore

        elif size is not None:
            return torch.empty(
                size,
                dtype=dtype,  # type: ignore
                device=device,
                memory_format=memory_format,
                out=out,
                layout=layout,
                pin_memory=pin_memory,
                requires_grad=requires_grad,
            )  # type: ignore

        else:
            msg = f"Internal error: found {data=} and {size=} in {cls.__name__}."
            raise RuntimeError(msg)


class Tensor0D(
    Generic[T_DType, T_Device],
    TTensor[_0DShape, T_DType, T_Device],
): ...

class FloatTensor(
    Generic[T_Shape, T_Device],
    TTensor[T_Shape, dtypes.FloatDType, T_Device],
): ...

class FloatTensor0D(
    Generic[T_Device],
    Tensor0D[dtypes.FloatDType, T_Device],
    FloatTensor[_0DShape, T_Device],
): ...

In [20]:
def merge_generics(gens: List[_TTensorGenerics]) -> _TTensorGenerics:
    result = _TTensorGenerics()
    for gen in gens:
        if gen.shape is not None:
            assert result.shape is None or result.shape == gen.shape
            result.shape = gen.shape
        if gen.dtype is not None:
            assert result.dtype is None or result.dtype == gen.dtype
            result.dtype = gen.dtype
        if gen.device is not None:
            assert result.device is None or result.device == gen.device
            result.device = gen.device
    return result

def dict_to_generics(gen_args_dict: Dict[str, TypeVar]) -> _TTensorGenerics:
    shape = _generic_shape_to_shape(gen_args_dict.get("T_Shape", Any))
    dtype = _generic_dtype_to_dtype(gen_args_dict.get("T_DType", Any))
    device = _generic_device_to_device(gen_args_dict.get("T_Device", Any))
    return _TTensorGenerics(shape, dtype, device)

def get_tensor_args(cls):
    # if cls is torch.Tensor:
    #     return _TTensorGenerics()

    if cls is TTensor or get_origin(cls) is TTensor:
        gen_args = get_args(cls)
        gen_args = gen_args + (_AnyShape, _AnyDType, _AnyDevice)[len(gen_args):]
        gen_shape, gen_dtype, gen_device = gen_args
        shape = _generic_shape_to_shape(gen_shape)
        dtype = _generic_dtype_to_dtype(gen_dtype)
        device = _generic_device_to_device(gen_device)
        result = _TTensorGenerics(shape, dtype, device)
        # print(f"get_tensor_args: from base {result=}")
        return result
    
    # if hasattr(cls, "_shape") and hasattr(cls, "_dtype") and hasattr(cls, "_device"):
    #     shape = cls._shape
    #     dtype = cls._dtype
    #     device = cls._device
    #     result = _TTensorGenerics(shape, dtype, device)
    #     print(f"get_tensor_args: from static {result=}")
    #     return result
    # raise RuntimeError

    origin = get_origin(cls)
    if origin is None:
        target_cls = cls
    else:
        target_cls = origin 
    print(f"DEBUG: {target_cls.__orig_bases__=}")

    generics = [basecls for basecls in target_cls.__orig_bases__ if get_origin(basecls) is Generic]  # type: ignore
    if len(generics) == 1:
        generic_names = [typevar.__name__ for typevar in get_args(generics[0]) if typevar in (T_Shape, T_DType, T_Device)]
    else:
        generic_names = []
    gen_args_dict = dict(zip(generic_names, get_args(cls)))
    cur_generics = dict_to_generics(gen_args_dict)
    # print(f"DEBUG: {cur_generics=}")

    # print(f"DEBUG: {cls=}")
    # print(f"DEBUG: {origin=}; {get_args(cls)=}")

    if origin is None:
        mro = inspect.getmro(cls)
    else:
        mro = inspect.getmro(origin)

    # print(f"DEBUG: {mro=}")
    # args_list = [get_args(base_cls) for base_cls in mro]
    # print(f"DEBUG: {args_list=}")
    # orig_bases_list = [getattr(base_cls, "__orig_bases__", None) for base_cls in mro]
    # print(f"DEBUG: {orig_bases_list=}")

    all_bases = [
        (i, base_cls_2)
        for i, base_cls in enumerate(mro)
        for base_cls_2 in getattr(base_cls, "__orig_bases__", ())
        if (base_cls_2 not in (torch.Tensor,) and get_origin(base_cls_2) is not Generic)
    ]
    # print(f"DEBUG: {all_bases=}")
    gens = [cur_generics] + [get_tensor_args(base_cls) for _, base_cls in all_bases]
    # print(f"DEBUG: {gens=}")
    # print("DEBUG: ===")
    return merge_generics(gens)


examples = [
    (TTensor, _TTensorGenerics()),
    (TTensor[_0DShape], _TTensorGenerics((), None, None)),
    (TTensor[Any, dtypes.Float16DType], _TTensorGenerics(None, torch.float16, None)),
    (TTensor[_1DShape, dtypes.DoubleDType, devices.CPUDeviceType], _TTensorGenerics((None,), torch.double, torch.device("cpu"))),
    (Tensor0D, _TTensorGenerics((), None, None)),
    (Tensor0D[dtypes.DoubleDType], _TTensorGenerics((), torch.double, None)),
    (FloatTensor0D, _TTensorGenerics((), torch.float, None)),
    (FloatTensor0D[devices.CUDADeviceType], _TTensorGenerics((), torch.float, torch.device("cuda"))),
]

print()
for i, (cls, expected) in enumerate(examples, start=1):
    print(f"{i}: {cls}")
    tensor_args = get_tensor_args(cls)
    check = tensor_args == expected
    if not check:
        print(f"{i}: {tensor_args} != {expected}")
    print(f"{i}: Sanity check: {check}")
    print(f"{i}: {cls()}")
    print()


1: <class '__main__.TTensor'>
1: Sanity check: True
__new__: <class '__main__.TTensor'>
1: tensor([])

2: __main__.TTensor[typing.Tuple[()]]
2: Sanity check: True
__new__: <class '__main__.TTensor'>
2: tensor([])

3: __main__.TTensor[typing.Any, torchwrench.core.dtype_enum_v2.Float16DType]
3: Sanity check: True
__new__: <class '__main__.TTensor'>
3: tensor([])

4: __main__.TTensor[typing.Tuple[int], torchwrench.core.dtype_enum_v2.Float64DType, torchwrench.core.device_enum.CPUDeviceType]
4: Sanity check: True
__new__: <class '__main__.TTensor'>
4: tensor([])

5: <class '__main__.Tensor0D'>
DEBUG: target_cls.__orig_bases__=(typing.Generic[+T_DType, +T_Device], __main__.TTensor[typing.Tuple[()], +T_DType, +T_Device])
5: Sanity check: True
__new__: <class '__main__.Tensor0D'>
5: tensor([])

6: __main__.Tensor0D[torchwrench.core.dtype_enum_v2.Float64DType]
DEBUG: target_cls.__orig_bases__=(typing.Generic[+T_DType, +T_Device], __main__.TTensor[typing.Tuple[()], +T_DType, +T_Device])
6: Sani

In [9]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

from typing import Any, Generic

from torchwrench.core import device_enum as devices
from torchwrench.core import dtype_enum_v2 as dtypes
# from torchwrench.types.typed_tensor.base import (
#     T_Device,
#     T_DType,
#     T_Shape,
#     TTensor,
#     _0DShape,
#     _1DShape,
#     _2DShape,
#     _3DShape,
# )

In [10]:
class Tensor1D(
    Generic[T_DType, T_Device],
    TTensor[_1DShape, T_DType, T_Device],
): ...


class Tensor2D(
    Generic[T_DType, T_Device],
    TTensor[_2DShape, T_DType, T_Device],
): ...


class Tensor3D(
    Generic[T_DType, T_Device],
    TTensor[_3DShape, T_DType, T_Device],
): ...


class BoolTensor(
    Generic[T_Shape, T_Device],
    TTensor[T_Shape, dtypes.BoolDType, T_Device],
): ...


class BoolTensor0D(
    Generic[T_Device],
    TTensor[_0DShape, dtypes.BoolDType, T_Device],
): ...


class CPUTensor(
    Generic[T_Shape, T_DType],
    TTensor[T_Shape, T_DType, devices.CPUDeviceType],
): ...


class CUDAFloatTensor2D(
    TTensor[_2DShape, dtypes.FloatDType, devices.CUDADeviceType],
): ...


class DoubleTensor(
    Generic[T_Shape, T_Device],
    TTensor[T_Shape, dtypes.DoubleDType, T_Device],
): ...


class FloatTensor(
    Generic[T_Shape, T_Device],
    TTensor[T_Shape, dtypes.FloatDType, T_Device],
): ...


class FloatTensor2D(
    Generic[T_Device],
    TTensor[_2DShape, dtypes.FloatDType, T_Device],
): ...


class HalfTensor(
    Generic[T_Shape, T_Device],
    TTensor[T_Shape, dtypes.HalfDType, T_Device],
): ...


class IntTensor(
    Generic[T_Shape, T_Device],
    TTensor[T_Shape, dtypes.IntDType, T_Device],
): ...


class LongTensor(
    Generic[T_Shape, T_Device],
    TTensor[T_Shape, dtypes.LongDType, T_Device],
): ...


class ShortTensor(
    Generic[T_Shape, T_Device],
    TTensor[T_Shape, dtypes.ShortDType, T_Device],
): ...


class SignedIntegerTensor(
    Generic[T_Shape, T_Device],
    TTensor[T_Shape, dtypes.DTypeBase[L[False], L[False], L[True]], T_Device],
): ...

__class_getitem__: <class '__main__.TTensor'>; (typing.Tuple[int], +T_DType, +T_Device)
__class_getitem__: <class '__main__.TTensor'>; (typing.Tuple[int, int], +T_DType, +T_Device)
__class_getitem__: <class '__main__.TTensor'>; (typing.Tuple[int, int, int], +T_DType, +T_Device)
__class_getitem__: <class '__main__.TTensor'>; (+T_Shape, <class 'torchwrench.core.dtype_enum_v2.BoolDType'>, +T_Device)
__class_getitem__: <class '__main__.TTensor'>; (typing.Tuple[()], <class 'torchwrench.core.dtype_enum_v2.BoolDType'>, +T_Device)
__class_getitem__: <class '__main__.TTensor'>; (+T_Shape, +T_DType, <class 'torchwrench.core.device_enum.CPUDeviceType'>)
__class_getitem__: <class '__main__.TTensor'>; (typing.Tuple[int, int], <class 'torchwrench.core.dtype_enum_v2.Float32DType'>, <class 'torchwrench.core.device_enum.CUDADeviceType'>)
__class_getitem__: <class '__main__.TTensor'>; (+T_Shape, <class 'torchwrench.core.dtype_enum_v2.Float64DType'>, +T_Device)
__class_getitem__: <class '__main__.TTensor

In [11]:
m = TTensor[_0DShape, dtypes.UInt32DType]()

# is_signed = x.is_signed()
is_signed = m.is_signed()
is_complex = m.is_complex()
is_floating_point = m.is_floating_point()

n = m.bool().item()

o = TTensor[Any, dtypes.BoolDType](1)
p = o.item()
q = o.int()

print(isinstance(o, SignedIntegerTensor))
print(isinstance(q, SignedIntegerTensor))


# TODO: rm
x = Tensor2D[dtypes.FloatDType, devices.CPUDeviceType]([[2, 3, 4], [5, 6, 7]])
m = x.ndim
z = x[0]
p = z.shape
a = z[0]
q = a.shape

y = x.view((3, 2, 1))
s = y.shape
n = y.ndim
o = y[0]
r = o[None]

b = x[None] == y

c = x.double()
d = x.short()

e = x.isfinite()
f = x.isinf()

g = x.mean()
h = x.sum().isinf()

i = g.ndim
j = h.ndim

k = x == c


__class_getitem__: <class '__main__.TTensor'>; (typing.Tuple[()], <class 'torchwrench.core.dtype_enum_v2.UInt32DType'>)
__new__: <class '__main__.TTensor'>


RuntimeError: a Tensor with 0 elements cannot be converted to Scalar